Instalação do Pydantic

In [2]:
!pip install pydantic
!pip install pydantic[email]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.1 MB/s eta 0:00:00


Importações das bibliotecas necessárias

In [3]:
from datetime import datetime
from typing import Optional
from uuid import uuid4

from fastapi import FastAPI
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, EmailStr, Field, field_serializer, UUID4

In [5]:
app = FastAPI()

Criação classe User (Usuário)

In [11]:
class User(BaseModel):
    model_config = {
        "extra": "forbid", # Proíbe a adição de atributos extras
    }
    __users__ = []
    name: str = Field(..., description="Name of the user")
    email: EmailStr = Field(..., description="Email address of the user") # Valida automaticamente se o valor é um e-mail válido
    friends: list[UUID4] = Field( # Lista de UUIDs, cada UUID representa um amigo
        default_factory=list, max_length=500, description="List of friends" # 'default_factory=list' cria uma nova lista vazia por instância
    )
    blocked: list[UUID4] = Field( # Lista de usuários bloqueados
        default_factory=list, max_length=500, description="List of blocked users"
    )
    signup_ts: Optional[datetime] = Field( # Campo opcional que representa quando o usuário se cadastrou
        default_factory=datetime.now, description="Signup timestamp", kw_only=True
    )
    id: UUID4 = Field( # Identificador único do usuário
        default_factory=uuid4, description="Unique identifier", kw_only=True # o kw_only serve para garantir que o campo só pode ser passado o valor caso ele seja declarado, ou seja, não recebe valor por posição do argumento
    )

    @field_serializer("id", when_used="json") # Define como o campo id será serializado quando convertido para JSON
    def serialize_id(self, id: UUID4) -> str:
        return str(id) # Converte o UUID para string


Criação de Endpoints

In [12]:
@app.get("/users", response_model=list[User]) # Cria um endpoint, sendo a resposta uma lista de usuários (Validada e serializada pelo modelo 'User')
async def get_users() -> list[User]:
    return list(User.__users__)


@app.post("/users", response_model=User) # Cria um endpoint, a reposta será um único objeto User
async def create_user(user: User):
    User.__users__.append(user) # Adiciona o novo usuário à lista em memória
    return user


@app.get("/users/{user_id}", response_model=User) # Cria um endpoint
async def get_user(user_id: UUID4) -> User | JSONResponse: # Recebe o user_id da URL, o FastAPI valida automaticamente se é um UUID válido
    try:
        return next((user for user in User.__users__ if user.id == user_id)) # Retorna o primeiro usuário cujo id seja igual ao user_id
    except StopIteration:
        return JSONResponse(status_code=404, content={"message": "User not found"})

Criação da função principal

In [13]:
def main() -> None:
    with TestClient(app) as client: # Cria um cliente de teste do FastAPI
        for i in range(5): # Cria 5 users
            response = client.post(
                "/users",
                json={"name": f"User {i}", "email": f"example{i}@arjancodes.com"},
            )
            assert response.status_code == 200
            assert response.json()["name"] == f"User {i}", (
                "The name of the user should be User {i}"
            )
            assert response.json()["id"], "The user should have an id"

            user = User.model_validate(response.json()) # Converte o JSON retornado em um objeto User validado e reaplica todas as regras do modelo Pydantic.
            assert str(user.id) == response.json()["id"], "The id should be the same"
            assert user.signup_ts, "The signup timestamp should be set"
            assert user.friends == [], "The friends list should be empty"
            assert user.blocked == [], "The blocked list should be empty"

        response = client.get("/users")
        assert response.status_code == 200, "Response code should be 200"
        assert len(response.json()) == 5, "There should be 5 users"

        response = client.post( # Cria um novo usuário manualmente
            "/users", json={"name": "User 5", "email": "example5@arjancodes.com"}
        )
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", (
            "The name of the user should be User 5"
        )
        assert response.json()["id"], "The user should have an id"

        user = User.model_validate(response.json()) # Valida novamente com o Pydantic agora
        assert str(user.id) == response.json()["id"], "The id should be the same"
        assert user.signup_ts, "The signup timestamp should be set"
        assert user.friends == [], "The friends list should be empty"
        assert user.blocked == [], "The blocked list should be empty"

        response = client.get(f"/users/{response.json()['id']}") # Busca usuário pelo id
        assert response.status_code == 200
        assert response.json()["name"] == "User 5", (
            "This should be the newly created user"
        )

        response = client.get(f"/users/{uuid4()}") # Gera um UUID aleatório que não existe
        assert response.status_code == 404
        assert response.json()["message"] == "User not found", (
            "We technically should not find this user"
        )

        response = client.post("/users", json={"name": "User 6", "email": "wrong"}) # Tenta criar usuário com e-mail inválido
        assert response.status_code == 422, "The email address is should be invalid"

In [14]:
if __name__ == "__main__":
    main()